In [1]:
from d2l import torch as d2l

In [2]:
def load_array(data_arrays, batch_size, is_train=True):
    """Construct a PyTorch data iterator.
    
    data_arrays: pytorch data, must have dimension 
    batch_size: the size of feature in each iter 
    is_train: shuffle or not
    """
    dataset = torch.utils.data.TensorDataset(*data_arrays)
    return torch.utils.data.DataLoader(dataset, batch_size, shuffle=is_train)

In [65]:
def save_checkpoint(epoch, net, optimizer, checkpoint_path):
    if type(checkpoint_path) is not str:
        checkpoint_path='checkpoint.pth'
    # 添加功能:匹配,加后缀 .pth 
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': net.module.state_dict() if isinstance(net,nn.DataParallel) \
                          else net.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }                        
    torch.save(checkpoint, checkpoint_path)

In [66]:
def train_GPU_save(net, train_iter, test_iter, num_epochs, lr, device, 
              save_checkpoints: str =True, use_checkpoints: str =None,
              save_net=False,
             ):
    """
    Train a model with a GPU .
    You can save checkpoints and the net . It's stronger.

    """
    def init_weights(m):
        if type(m) == nn.Linear or type(m) == nn.Conv2d:
            nn.init.xavier_uniform_(m.weight)
    net.apply(init_weights)
    print('training on', device)

    optimizer = torch.optim.SGD(net.parameters(), lr=lr)
    loss = nn.CrossEntropyLoss()
    start_epoch=0
    
    animator = d2l.Animator(xlabel='epoch', xlim=[1, num_epochs],
                            legend=['train loss', 'train acc', 'test acc'])
    timer, num_batches = d2l.Timer(), len(train_iter)
            
    if use_checkpoints:
        try:
            checkpoint = torch.load(use_checkpoints, weights_only=True)
            if isinstance(net, nn.DataParallel):
                net.module.load_state_dict(checkpoint['model_state_dict'])
            else:
                net.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch=checkpoint['epoch']+1
            print('Use checkpoint successfully!')
        except FileNotFoundError:
            print('ATTENTION: There is not the checkpoint.')
        except Exception:
            raise
    
    net.to(device)
    for epoch in range(start_epoch,num_epochs):
        # Sum of training loss, sum of training accuracy, no. of examples
        metric = d2l.Accumulator(3)
        net.train()
        for i, (X, y) in enumerate(train_iter):
            timer.start()
            optimizer.zero_grad()
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            l = loss(y_hat, y)
            l.backward()
            optimizer.step()
            with torch.no_grad():
                metric.add(l * X.shape[0], d2l.accuracy(y_hat, y), X.shape[0])
            timer.stop()
            train_l = metric[0] / metric[2]
            train_acc = metric[1] / metric[2]
            if (i + 1) % (num_batches // 5) == 0 or i == num_batches - 1:
                animator.add(epoch + (i + 1) / num_batches,
                             (train_l, train_acc, None))
        
        if epoch%round(num_epochs*0.25)==0 and save_checkpoints:
           save_checkpoint(epoch, net, optimizer, save_checkpoints)
        
        test_acc = d2l.evaluate_accuracy_gpu(net, test_iter)
        animator.add(epoch + 1, (None, None, test_acc))
    
    if save_net:
        pass
        
    print(f'loss {train_l:.3f}, train acc {train_acc:.3f}, '
          f'test acc {test_acc:.3f}')
    print(f'{metric[2] * num_epochs / timer.sum():.1f} examples/sec \n'
          f'total time:{timer.sum():.4f} sec '
          f'on {str(device)}')

In [2]:
def draw_parameter_BN(net):
    '''
    draw gamma and beta of the BatchNorm layer
    also can be used for others
    '''
    import matplotlib.pyplot as plt
    %matplotlib inline
    import math
    count=0
    weight=[]
    bias=[]
    for layer in net:
        if isinstance(layer,(nn.BatchNorm1d, nn.BatchNorm2d)):
            count+=1
            layer_dict=dict(layer.named_parameters())
            weight.append(layer_dict['weight'].data)
            bias.append(layer_dict['bias'].data)
    fig,ax=plt.subplots(math.ceil(count/2),2,figsize=(6*2,4*math.ceil(count/2)),sharey=True,)
    for i in range(count):
        weight_i=weight[i]
        bias_i=bias[i]
        x=torch.arange(weight_i.shape[0])
        ax[i//2,i%2].scatter(x=x,y=weight_i,label='weight')
        ax[i//2,i%2].plot(weight_i.mean().expand(weight_i.shape[0]),color='pink',label='mean_weight')
        ax[i//2,i%2].scatter(x=x,y=bias_i,color='purple',label='bias')
        ax[i//2,i%2].plot(bias_i.mean().expand(bias_i.shape[0]),color='red',linestyle='dotted',label='mean_bias')
        ax[i//2,i%2].text(0,5, f"std of weight {weight_i.std():.2f}",family="monospace", fontsize=10)
        ax[i//2,i%2].text(0,4.5, f"std of bias {bias_i.std():.2f}",family="monospace", fontsize=10)
    
        ax[i//2,i%2].legend()